# NeuroGPT Baseline: 4-class Motor Imagery (BCI Competition IV-2a)

**Goal:** reproduce the published NeuroGPT fine-tuning recipe (EEG encoder + GPT-2, pretrained on TUH EEG) on BCI-IV-2a, with **no modifications and no novelty**. This is the reference our prompt-tuning method is compared against.

| Item | Value |
|---|---|
| Model | Neuro-GPT (6-layer EEG encoder + 6-layer GPT-2 decoder, 1024-dim) |
| Pretrained weights | `wenhuic/Neuro-GPT` on HuggingFace |
| Task | Left hand / right hand / feet / tongue motor imagery |
| Split | Fold 0 = leave-one-subject-out: train on subjects A02-A09 (16 sessions), test on **unseen subject A01** (2 sessions, 576 trials) |
| Recipe | `--ft-only-encoder=True`, 10,000 steps, batch 32, lr 1e-4 |
| Hardware | Colab T4 GPU |

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
torch 2.11.0+cu130 | CUDA available: True


## 2. Clone the original repo and pin its dependencies

The vendor code targets `transformers==4.38.1`. Colab ships much newer versions that break it (`transformers.deepspeed` removed, `accelerate` API changed, `peft` incompatible), so we pin the versions the repo was written for. `peft` is unused by NeuroGPT and is removed.

In [ ]:
%cd /content
!rm -rf NeuroGPT && git clone -q https://github.com/wenhui0206/NeuroGPT.git
!pip install -q mne torchinfo "transformers==4.38.1" "tokenizers<0.19" "accelerate==0.27.2" 2>&1 | grep -v "requires\|dependency resolver" | tail -3
!pip uninstall -y -q peft 2>/dev/null; true
!cd NeuroGPT/src && python -c "import transformers, accelerate; from trainer.make import make_trainer; print('transformers', transformers.__version__, '| accelerate', accelerate.__version__, '| vendor imports OK')" 2>&1 | tail -1

/content
transformers 4.38.1 | accelerate 0.27.2 | vendor imports OK


## 3. Download the pretrained checkpoint (~303 MB)

Released by the NeuroGPT authors: https://huggingface.co/wenhuic/Neuro-GPT

In [ ]:
!mkdir -p /content/NeuroGPT/pretrained_model
!wget -q -O /content/NeuroGPT/pretrained_model/pytorch_model.bin https://huggingface.co/wenhuic/Neuro-GPT/resolve/main/pretrained_model/pytorch_model.bin
!ls -l /content/NeuroGPT/pretrained_model/   # expect 318189905 bytes

total 310736
-rw-r--r-- 1 root root 318189905 Oct  1 21:10 pytorch_model.bin


## 4. Download BCI Competition IV-2a

9 subjects, 2 sessions each (`T` = training session, `E` = evaluation session), 22 EEG channels at 250 Hz, 288 trials per session, plus the true class labels for each session. Source: https://www.bbci.de/competition/iv/#dataset2a

In [ ]:
!rm -rf /content/raw_bci2a && mkdir -p /content/raw_bci2a
%cd /content/raw_bci2a
!wget -q https://www.bbci.de/competition/download/competition_iv/BCICIV_2a_gdf.zip -O gdf.zip
!wget -q https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip -O labels.zip
!unzip -oq gdf.zip && unzip -oq labels.zip -d true_labels_tmp
!echo "GDF recordings:" $(ls *.gdf | wc -l) "| label files:" $(ls true_labels_tmp | wc -l)

/content/raw_bci2a
GDF recordings: 18 | label files: 18


## 5. Convert GDF to the `.npz` format the repo expects

`MotorImageryDataset` reads, per recording, an `.npz` with keys `s` (signal, µV, shape `time x channels`), `etyp`/`epos`/`edur` (event code / onset / duration in samples) and `artifacts`, plus a `true_labels/<name>.mat` file. The repo ships no converter, so we build one with MNE. Each session should contain **288** trial-start events (code 768).

In [ ]:
import os, shutil, warnings, numpy as np, mne
warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

RAW, OUT = "/content/raw_bci2a", "/content/bci2a_egg_npz"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT + "/true_labels")

for f in sorted(x for x in os.listdir(RAW) if x.endswith(".gdf")):
    raw = mne.io.read_raw_gdf(f"{RAW}/{f}", preload=True)
    sf = raw.info["sfreq"]
    etyp = np.array([int(d) for d in raw.annotations.description]).reshape(-1, 1)
    np.savez(
        f"{OUT}/{f[:-4]}.npz",
        s=(raw.get_data() * 1e6).T,
        etyp=etyp,
        epos=np.round(raw.annotations.onset * sf).astype(int).reshape(-1, 1),
        edur=np.round(raw.annotations.duration * sf).astype(int).reshape(-1, 1),
        artifacts=np.zeros((len(etyp), 1)),
    )
    print(f"{f[:-4]}: {raw.n_times} samples @ {sf:.0f} Hz, {int((etyp == 768).sum())} trials")

for f in os.listdir(RAW + "/true_labels_tmp"):
    shutil.copy(f"{RAW}/true_labels_tmp/{f}", f"{OUT}/true_labels/{f}")
print("\nconverted:", len([x for x in os.listdir(OUT) if x.endswith('.npz')]), "files ->", OUT)

## 6. Training configuration

This is the repo's own `scripts/finetune.sh` recipe: **fine-tune the encoder only** (`--ft-only-encoder=True`), pretrained weights loaded, 4-class decoding head, fold 0.

In [ ]:
RUN_NAME, LOG_DIR = "actual_baseline_colab", "/content/results"
!rm -rf {LOG_DIR}

args = {
    "training-style": "decoding", "num-decoding-classes": 4,
    "training-steps": 10000, "eval_every_n_steps": 500, "log-every-n-steps": 1000,
    "num_chunks": 2, "chunk_len": 500, "chunk_ovlp": 0,
    "per-device-training-batch-size": 32, "per-device-validation-batch-size": 32,
    "ft-only-encoder": True, "use-encoder": True, "fold_i": 0,
    "num-encoder-layers": 6, "num-hidden-layers": 6, "embedding-dim": 1024,
    "learning-rate": 1e-4, "run-name": RUN_NAME,
    "pretrained-model": "../pretrained_model/pytorch_model.bin",
    "dst-data-path": "/content/bci2a_egg_npz/", "log-dir": LOG_DIR,
}
CMD = "python ../src/train_gpt.py " + " ".join(f"--{k}={v}" for k, v in args.items())
for k, v in args.items():
    print(f"{k:35s} {v}")

training-style                      decoding
num-decoding-classes                4
training-steps                      10000
eval_every_n_steps                  500
log-every-n-steps                   1000
num_chunks                          2
chunk_len                           500
chunk_ovlp                          0
per-device-training-batch-size      32
per-device-validation-batch-size    32
ft-only-encoder                     True
use-encoder                         True
fold_i                              0
num-encoder-layers                  6
num-hidden-layers                   6
embedding-dim                       1024
learning-rate                       0.0001
run-name                            actual_baseline_colab
pretrained-model                    ../pretrained_model/pytorch_model.bin
dst-data-path                       /content/bci2a_egg_npz/
log-dir                             /content/results


## 7. Launch training

Runs `src/train_gpt.py` **unmodified** in the background (about 12 min on a T4) and writes to `train.log`, so the next cell can track it.

In [ ]:
import subprocess
proc = subprocess.Popen(f"cd /content/NeuroGPT/scripts && {CMD} > /content/train.log 2>&1", shell=True)
print("training started, pid", proc.pid)

training started, pid 15147


## 8. Training progress

Every 500 steps the model is evaluated on the **unseen subject A01** (never used in training). The cell waits for training to finish and prints one row per evaluation.

In [ ]:
import re, time
EVAL = re.compile(r"'eval_loss': ([\d.]+), 'eval_accuracy': ([\d.]+).*?'epoch': ([\d.]+)")
shown = 0
while True:
    done = proc.poll() is not None
    for loss, acc, ep in EVAL.findall(open("/content/train.log").read())[shown:]:
        shown += 1
        print(f"eval {shown:2d}  step {shown*500:5d}  epoch {float(ep):5.2f}  held-out loss {float(loss):.3f}  acc {float(acc):.3f}")
    if done:
        break
    time.sleep(15)
print("\ntraining finished, exit code", proc.returncode)

eval  1  step   500  held-out loss 1.301  acc 0.469
eval  2  step  1000  held-out loss 1.061  acc 0.568
eval  3  step  1500  held-out loss 0.922  acc 0.635
eval  4  step  2000  held-out loss 0.900  acc 0.637
eval  5  step  2500  held-out loss 0.936  acc 0.630
eval  6  step  3000  held-out loss 0.889  acc 0.639
eval  7  step  3500  held-out loss 0.914  acc 0.639
eval  8  step  4000  held-out loss 0.896  acc 0.653
eval  9  step  4500  held-out loss 0.910  acc 0.653
eval 10  step  5000  held-out loss 0.924  acc 0.660
eval 11  step  5500  held-out loss 0.894  acc 0.656
eval 12  step  6000  held-out loss 0.958  acc 0.667
eval 13  step  6500  held-out loss 0.894  acc 0.665
eval 14  step  7000  held-out loss 0.941  acc 0.658
eval 15  step  7500  held-out loss 0.910  acc 0.656
eval 16  step  8000  held-out loss 0.895  acc 0.667
eval 17  step  8500  held-out loss 0.918  acc 0.660
eval 18  step  9000  held-out loss 0.975  acc 0.644
eval 19  step  9500  held-out loss 0.937  acc 0.656
eval 20  ste

## 9. Results

**Held-out accuracy** is measured on subject A01 (2 sessions), who is not used for training. This is cross-subject generalisation, which is much harder than within-subject classification.

> **Caveat on the repo's own `test_metrics.csv`:** in decoding mode `train_gpt.py` swaps the datasets, so its final "test" step runs on the *training* subjects. That number is inflated and is shown below only for transparency. The held-out accuracy is the one to report.

**Reading the curves:** held-out accuracy plateaus near 0.65 from about step 3,000 while training accuracy is far higher, so the model is overfitting to the 8 training subjects. This is one fold (one test subject), so the number is noisy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ev = EVAL.findall(open("/content/train.log").read())
steps = np.arange(1, len(ev) + 1) * 500
acc = np.array([float(a) for _, a, _ in ev]); loss = np.array([float(l) for l, _, _ in ev])

d = f"{LOG_DIR}/{RUN_NAME}-0"
p, y = np.load(f"{d}/test_predictions.npy"), np.load(f"{d}/test_label_ids.npy")
print(pd.DataFrame({
    "metric": ["held-out accuracy (final, step 10000)", "held-out accuracy (best checkpoint)",
               "chance level (4 classes)", "repo 'test' accuracy (TRAINING subjects, inflated)"],
    "value": [acc[-1], acc.max(), 0.25, (p.argmax(-1) == y).mean()],
}).round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(steps, acc, marker="o"); ax[0].axhline(0.25, ls="--", c="gray", label="chance")
ax[0].set(title="Held-out accuracy", xlabel="step", ylabel="accuracy"); ax[0].legend()
ax[1].plot(steps, loss, marker="o", c="tab:red"); ax[1].set(title="Held-out loss", xlabel="step")
plt.tight_layout(); plt.show()

                                            metric  value
             held-out accuracy (final, step 10000)  0.648
               held-out accuracy (best checkpoint)  0.667
                          chance level (4 classes)  0.250
repo 'test' accuracy (TRAINING subjects, inflated)  0.894

[plot: results/baseline_fold0_curve.png]


## 10. All 9 folds (leave-one-subject-out)

The paper reports the **mean over 9 leave-one-subject-out folds** (Table 1: encoder-only with pre-training = **0.645 ± 0.104**). One fold is one test subject and is very noisy, so we run the remaining folds 1 to 8 with the identical recipe (fold 0 was run above).

Fold `i` holds out subject `A0(i+1)` (both sessions) and trains on the other 8 subjects. Two folds run concurrently on the one GPU to save time, and each finished fold is copied to Google Drive so nothing is lost if the runtime disconnects.

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/neurogpt_results"
os.makedirs(DRIVE_OUT, exist_ok=True)

def fold_cmd(f):
    a = dict(args, fold_i=f)
    return "python ../src/train_gpt.py " + " ".join(f"--{k}={v}" for k, v in a.items())

def chain(folds):
    steps = [f"({fold_cmd(f)} > /content/train_fold{f}.log 2>&1; "
             f"cp -r {LOG_DIR}/{RUN_NAME}-{f} {DRIVE_OUT}/ 2>/dev/null; cp /content/train_fold{f}.log {DRIVE_OUT}/ 2>/dev/null)"
             for f in folds]
    return subprocess.Popen("cd /content/NeuroGPT/scripts && " + " ; ".join(steps), shell=True)

fold_procs = [chain([1, 3, 5, 7]), chain([2, 4, 6, 8])]
print("started 2 concurrent chains: folds 1,3,5,7 and 2,4,6,8")

### Progress

Shows each fold's current step and its latest held-out accuracy. Re-run this cell to refresh; it waits up to 20 minutes or until all folds finish.

In [ ]:
import glob
STEP = re.compile(r"(\d+)/10000 \[")
def fold_status(f):
    p = f"/content/train_fold{f}.log"
    if not os.path.exists(p): return "queued"
    t = open(p, errors="ignore").read()
    ev = EVAL.findall(t)
    steps = STEP.findall(t[-3000:])
    done = os.path.exists(f"{LOG_DIR}/{RUN_NAME}-{f}/test_metrics.csv")
    last = f"{float(ev[-1][1]):.3f}" if ev else "-"
    return f"{'DONE' if done else 'running'}  step {steps[-1] if steps else '?':>5}  evals {len(ev):2d}  latest held-out acc {last}"

t0 = time.time()
while time.time() - t0 < 1200 and any(p.poll() is None for p in fold_procs):
    time.sleep(30)
for f in range(1, 9):
    print(f"fold {f} (hold out A0{f+1}):", fold_status(f))

## 11. Summary over all 9 folds vs the paper

Per-fold accuracy is the held-out accuracy **at the final step (10,000)** on the unseen subject, not the best checkpoint (choosing the best checkpoint on the test subject would leak information). Mean and standard deviation are taken across the 9 subjects. The std is the sample std (ddof = 1); the paper does not say which it uses.

In [ ]:
def fold_log(f):
    return "/content/train.log" if f == 0 else f"/content/train_fold{f}.log"

rows = []
for f in range(9):
    ev = [float(a) for _, a, _ in EVAL.findall(open(fold_log(f), errors="ignore").read())]
    rows.append({"fold": f, "held_out_subject": f"A0{f+1}", "final_acc": ev[-1], "best_acc": max(ev), "n_evals": len(ev)})
df = pd.DataFrame(rows)
print(df.round(3).to_string(index=False))

m, s = df.final_acc.mean(), df.final_acc.std(ddof=1)
print(f"\nOurs   (encoder-only, pre-trained, 9-fold LOSO): {m:.3f} +/- {s:.3f}")
print("Paper  (encoder-only, pre-trained, Table 1)      : 0.645 +/- 0.104")
print("Paper  (encoder-only, from scratch)              : 0.606 +/- 0.098")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(df.held_out_subject, df.final_acc, color="tab:blue")
ax.axhline(m, c="tab:blue", ls="-", label=f"ours mean {m:.3f}")
ax.axhline(0.645, c="tab:green", ls="--", label="paper mean 0.645")
ax.axhline(0.25, c="gray", ls=":", label="chance 0.25")
ax.set(ylim=(0, 1), ylabel="held-out accuracy", xlabel="held-out subject", title="Neuro-GPT encoder-only baseline, leave-one-subject-out")
ax.legend(loc="upper right"); plt.tight_layout(); plt.show()

 fold held_out_subject  final_acc  best_acc  n_evals
    0              A01      0.648     0.667       20
    1              A02      0.438     0.464       20
    2              A03      0.743     0.750       20
    3              A04      0.507     0.528       20
    4              A05      0.562     0.608       20
    5              A06      0.498     0.530       20
    6              A07      0.696     0.717       20
    7              A08      0.759     0.762       20
    8              A09      0.615     0.632       20

Ours   (encoder-only, pre-trained, 9-fold LOSO): 0.607 +/- 0.114
Paper  (encoder-only, pre-trained, Table 1)      : 0.645 +/- 0.104
Paper  (encoder-only, from scratch)              : 0.606 +/- 0.098

[plot: results/baseline_9fold.png]


## 12. Auto-save

Copies the per-fold results, logs and the summary table to Google Drive (`MyDrive/neurogpt_results/`) so they survive a runtime reset.

In [ ]:
df.to_csv(f"{DRIVE_OUT}/baseline_9fold_summary.csv", index=False)
df.to_csv(f"{LOG_DIR}/baseline_9fold_summary.csv", index=False)
for f in range(9):
    shutil.copytree(f"{LOG_DIR}/{RUN_NAME}-{f}", f"{DRIVE_OUT}/{RUN_NAME}-{f}", dirs_exist_ok=True)
    shutil.copy(fold_log(f), f"{DRIVE_OUT}/train_fold{f}.log")
!ls {DRIVE_OUT}

actual_baseline_colab-0 ... actual_baseline_colab-8, train_fold0.log ... train_fold8.log, baseline_9fold_summary.csv
